In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ f:\mathbb{R} \to \mathbb{R}, \quad y=L(p) = -t\ln(p)-(1-t)\ln(1-p) $$

$$ f:\mathbb{R^n} \to \mathbb{R^n}, \quad \mathbf{y} = L(\mathbf{p}) = (f(p_1), f(p_2), \dots, f(p_n)) $$

$$ \frac{\partial f}{\partial p_i} = -t_i\frac{1}{p_i} + (1-t_i)\frac{1}{1-p_i} = \frac{p_i-t_i}{p_i(1-p_i)} $$

$$
J_f(\mathbf{p}) =
\begin{bmatrix}
    \frac{\partial f}{\partial p_1} & 0 & \cdots & 0 \\
    0 & \frac{\partial f}{\partial p_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{\partial f}{\partial p_n}
\end{bmatrix}
$$

$$ df = J_f(\mathbf{p}) \cdot d\mathbf{p} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{df}, df \right \rangle =
\left \langle \frac{dF}{d\mathbf{p}}, d\mathbf{p} \right \rangle
$$

$$
\left \langle \frac{dF}{df}, J_f(\mathbf{p}) \cdot d\mathbf{p} \right \rangle =
\left \langle \frac{dF}{d\mathbf{p}}, d\mathbf{p} \right \rangle \implies
$$

$$ \frac{dF}{d\mathbf{p}} = J_f(\mathbf{p})^\top \, \frac{dF}{df} $$

$$ \frac{dF}{d\mathbf{p}} = \frac{df}{d\mathbf{p}} \odot \frac{dF}{df} $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def _bce(p: tr.Tensor, t: tr.Tensor, reduction = "mean") -> tr.Tensor:
    """
    Computes the binary cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function assumes that `p` has already been clamped to avoid log(0) issues.
    """

    L = - (t * p.log() + (1 - t) * (1 - p).log())
    
    if reduction != "mean":
        assert False, f"Unsupported reduction: {reduction}"
        
    L = L.mean()
    return L


def bce(p: tr.Tensor, t: tr.Tensor, reduction = "mean") -> tr.Tensor:
    """
    Computes the binary cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function clamps `p` to avoid log(0) issues.
    """

    assert tr.all((p >= -1.0) & (p <= +1.0))

    eps = 1e-7
    pc = p.clamp(eps, 1 - eps)
    L = _bce(pc, t, reduction)
    return L


class BCEFunction(autograd.Function):
    """Custom autograd function for the binary cross-entropy loss."""

    @staticmethod
    def forward(ctx, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        eps = 1e-7
        pc = p.clamp(eps, 1 - eps)
        ctx.save_for_backward(pc, t)
        y = _bce(pc, t)
        return y

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, None]:        
        (p, t) = ctx.saved_tensors
        S = p.shape[0]
        FO = p.shape[1]
        N = S * FO

        df_dp = 1/N * (p - t) / (p * (1 - p))         
        df_dp = df_dp * dF_df

        return (df_dp, None)
   

class BCE(nn.Module):
    """Custom module for the binary cross-entropy loss."""

    def __init__(self, reduction: str = "mean"):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"
        
    def forward(self, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return BCEFunction.apply(p, t)

In [ ]:
# Unit tests

def test_gradcheck(samples=10):
    def fn(p, y):
        return BCEFunction.apply(p, y)

    p = tr.rand(samples, 1, dtype=tr.float64, requires_grad=True)
    y = tr.randint(0, 2, (samples, 1)).float().requires_grad_(False)
    assert autograd.gradcheck(fn, (p, y), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    p = tr.rand(samples, 1, requires_grad=True)
    y = tr.randint(0, 2, (samples, 1)).float()

    p1 = p.clone().detach().requires_grad_(True)
    y1 = BCE(reduction='mean')(p1, y)
    y1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    y2 = nn.BCELoss(reduction='mean')(p2, y)
    y2.backward()

    assert y1.item() == approx(y2.item())
    assert p1.grad == approx(p2.grad)


if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)